# §35 — Zincirleme: kapasite × yazım kuralı (küçük ölçek, CPU)

**Merkezi soru:** state neden **bir** sınırı geçiyor da **ikincisini** geçemiyor?
(§26b: K=0'da %33-53, K=2'de şansa düşüyor.)

**Hipotez — projenin kendi bulgusundan:** bellek **girişim-sınırlı** (README §0;
§27a'da η taramasıyla bağımsız doğrulandı — platoyu uzatmak girişimi biriktirip
*kötüleştirdi*). Dolgu chunk'ları her 64 tokende distraktör kv yazıyor; iki chunk
sonra hedef gömülüyor.

**İki kaldıraç (ikisi de O(1)'i BOZMAZ):**
- **Kapasite:** `dpfp_nu` 2→8 ⇒ `key_dim` 256→1024, state **4×**. Sabit büyür,
  bağlamla değil. (Not: `bulk_dim` belleği etkilemiyor, o FFN'de.)
- **Yazım kuralı:** `additive` (biriktir) → `delta` (eskiyi sil, üzerine yaz).

**Tasarım:** 2×2 kol × çok seed, K ∈ {0,1,2,4,8}.
**Birincil metrik:** K=2'de eşleşmiş sonda doğruluğu (şu an çöktüğü nokta).
**İkincil:** K=0 regresyonu, eğitim kaybı.

**Ön-kayıtlı hüküm** (şans %3.3):
- **ZİNCİRLEME AÇILDI:** bir kol K=2'de **≥%30** VE K=0'da regresyon yok
- **KISMİ:** K=2'de %10-30
- **ETKİ YOK:** hepsi K=2'de <%10 → girişim de değil; sınır okuma/adresleme yolunda

**GPU gerekmez.** Kol başına ~10-20 dk (CPU).


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, itertools
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
CKDIR = os.path.join(BASE,'chain35'); os.makedirs(CKDIR, exist_ok=True)
try:
    from google.colab import drive; drive.mount('/content/drive')
    CKDIR = '/content/drive/MyDrive/hfp_chain35'; os.makedirs(CKDIR, exist_ok=True)
except Exception as e:
    print(f'Drive yok ({type(e).__name__}) -> yerel: {CKDIR}')
print('repo:', REPO, '| ckpt:', CKDIR)
for nu in (2,8): print(f'  dpfp_nu={nu} -> key_dim={2*64*nu}, M=({2*64*nu},64)={2*64*nu*64:,} float/katman')

In [ ]:
# --- 2. KOLLAR: 2x2 (kapasite x yazim) x seed ---
SEEDS = [0,1,2,3]
ARMS = [(2,'additive','taban (mevcut)'), (8,'additive','4x state'),
        (2,'delta','delta yazim'),      (8,'delta','4x state + delta')]
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': CKDIR,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'30'}
MODE = 'exp'                      # tek degisken kapasite/yazim olsun (retention sabit)
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = '' if (nu==2 and wr=='additive') else f'_nu{nu}{wr[0]}'
        if os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}.pt'):
            print(f'[atla] {label} s{s}'); continue
        print(f'\n=== EGITIM {label} (nu={nu}, {wr}) s{s} ===', flush=True)
        env = {**BASE_ENV, 'CC_DPFP_NU': str(nu), 'CC_WRITE': wr}
        subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',MODE,str(s),'6000'],
                       cwd=REPO, env=env)
print('\nEGITIM TAMAM')

In [ ]:
# --- 3. EVAL: eslesmis sonda, K in {0,1,2,4,8} ---
import csv
KS = [0,1,2,4,8]
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = '' if (nu==2 and wr=='additive') else f'_nu{nu}{wr[0]}'
        if not os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}.pt'):
            print(f'[eksik ckpt] {label} s{s}'); continue
        if os.path.exists(f'{CKDIR}/matchedv1{cfg}_{MODE}_s{s}.csv'):
            continue
        env = {**BASE_ENV, 'MP_DPFP_NU': str(nu), 'MP_WRITE': wr,
               'MP_KS': ','.join(map(str,KS)), 'MP_TRIALS':'100'}
        print(f'\n=== EVAL {label} s{s} ===', flush=True)
        subprocess.run([sys.executable,'review_scripts/matched_probe.py',MODE,str(s)],
                       cwd=REPO, env=env)
print('\nEVAL TAMAM')

In [ ]:
# --- 4. TOPLA + ON-KAYITLI HUKUM (§35) ---
import csv, statistics as st
def load(nu, wr):
    cfg = '' if (nu==2 and wr=='additive') else f'_nu{nu}{wr[0]}'
    acc = {k: [] for k in KS}
    for s in SEEDS:
        f = f'{CKDIR}/matchedv1{cfg}_{MODE}_s{s}.csv'
        if not os.path.exists(f): continue
        for r in csv.DictReader(open(f)):
            k = int(r['K'])
            if k in acc: acc[k].append(float(r['matched_acc']))
    return acc

print(f"{'kol':>22} " + ' '.join(f'K={k:<7}' for k in KS))
print('-'*74)
res = {}
for nu, wr, label in ARMS:
    a = load(nu, wr); res[(nu,wr)] = a
    row = []
    for k in KS:
        row.append(f'{st.mean(a[k]):5.1f}%  ' if a[k] else '  —    ')
    print(f'{label:>22} ' + ' '.join(row))
print(f'\n(sans %3.3 | n={len(SEEDS)} seed x 100 deneme)')

base = res[(2,'additive')]
print('\n=== ON-KAYITLI HUKUM (§35) ===')
best = None
for nu, wr, label in ARMS:
    if (nu,wr) == (2,'additive'): continue
    a = res[(nu,wr)]
    if not a[2] or not base[2]: continue
    k2, k0 = st.mean(a[2]), st.mean(a[0])
    reg = st.mean(base[0]) - k0            # K=0'da regresyon var mi
    tag = f'{label}: K=2 {k2:.1f}% (taban {st.mean(base[2]):.1f}%), K=0 {k0:.1f}% (taban {st.mean(base[0]):.1f}%)'
    if k2 >= 30 and reg <= 10:
        print(f'  ZINCIRLEME ACILDI -> {tag}')
        best = best or label
    elif k2 >= 10:
        print(f'  KISMI -> {tag}')
    else:
        print(f'  etki yok -> {tag}')
if best:
    print(f'\n=> GIRISIM HIPOTEZI DOGRULANDI ({best}). Kaldirac belli;')
    print('   sonraki: menzil genisletme (K=8,16,32) — nerede doyuyor?')
elif any(st.mean(res[a[:2]][2]) >= 10 for a in ARMS if res[a[:2]][2] and a[:2]!=(2,'additive')):
    print('\n=> KISMI: yon dogru ama esik alti. Daha agresif kapasite (nu=16) denenir.')
else:
    print('\n=> ETKI YOK: girisim de degil. Sinir OKUMA/ADRESLEME yolunda.')
    print('   Sonraki: yazma mi okuma mi bozuk? (ayni chunk icinde oku vs sinirdan sonra oku)')